# Project Canary — Harvest-Recovery Forecast

**Purpose:** reproduce the complete forecasting workflow in a form the capstone team can run and defend.

**Business question:** using only records known on a review date, what last-recorded harvest-recovery proxy should we expect for this building, compared with the 95% goal?

This notebook does not calculate the independent rules-based risk score and does not prove that any input causes recovery to change.

## 1. Define Y, X, and the unit of analysis

- **Y target:** population on the building's last recorded daily date ÷ beginning population.
- **Important:** this is the agreed capstone recovery proxy, not a verified harvest-event label.
- **One outcome:** one building in one completed cycle.
- **One training snapshot:** that building's facts known at a selected age. To avoid overweighting long cycles, training retains Days 7, 14, 21, 28, plus the last eligible pre-outcome snapshot.
- **Candidate X inputs:** production age; current survival; mortality level and recent trend; feed; latest weight evidence; and available recent temperature/humidity summaries. The selected compact model deliberately excludes building identity and raw inventory size.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "canary").exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "FARM HARVEST DATA.xlsx"
MODEL_READY_DIR = ROOT / "outputs" / "model_ready"

from canary import load_workbook

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
dataset = load_workbook(DATA_PATH)
print(f"Source: {dataset.source_name}")
print(f"Canonical building-day rows: {len(dataset.daily):,}")
print(f"Recorded building-cycles: {len(dataset.cycles):,}")
print(f"Blocking data-quality checks passed: {dataset.quality.passed}")
print(f"Non-blocking warnings: {len(dataset.quality.warnings)}")

Source: FARM HARVEST DATA.xlsx
Canonical building-day rows: 1,666
Recorded building-cycles: 34
Blocking data-quality checks passed: True
Non-blocking warnings: 3


In [2]:
from canary import build_modeling_snapshots, build_recovery_training_snapshots, train_outcome_model

daily_snapshots = build_modeling_snapshots(dataset, "recovery")
training_snapshots = build_recovery_training_snapshots(dataset)
coverage = pd.DataFrame({
    "Measure": ["Complete cycles", "Distinct building outcomes", "All leakage-safe daily snapshots", "Balanced decision snapshots"],
    "Count": [training_snapshots["cycle_id"].nunique(), training_snapshots[["cycle_id", "building_id"]].drop_duplicates().shape[0], len(daily_snapshots), len(training_snapshots)],
})
coverage

,Measure,Count
0,Complete cycles,5
1,Distinct building outcomes,25
2,All leakage-safe daily snapshots,1122
3,Balanced decision snapshots,122


In [3]:
exported = pd.read_csv(MODEL_READY_DIR / "recovery_training.csv")
assert len(exported) == len(training_snapshots) == 122
expected_keys = set(zip(training_snapshots.cycle_id.astype(str), training_snapshots.building_id, training_snapshots.as_of_date.astype(str)))
exported_keys = set(zip(exported.cycle_id.astype(str), exported.building_id, exported.as_of_date.astype(str)))
assert exported_keys == expected_keys
print("Export reconciliation passed: the CSV contains the exact 122 balanced recovery snapshots.")

Export reconciliation passed: the CSV contains the exact 122 balanced recovery snapshots.


## 2. Preprocessing and validation

1. Convert the workbook to one canonical building-day row; zone rows are aggregated before modeling.
2. Construct every snapshot with records dated on or before its review date; later records are excluded.
3. Median-impute missing numeric inputs inside each training fold. Ridge also adds missingness indicators and standardizes inputs.
4. Use **leave-one-complete-cycle-out cross-validation**: train on all but one cycle and test on the unseen cycle. This is the appropriate grouped equivalent of K-fold CV here.
5. Compare candidates primarily on **cycle-macro MAE** so each cycle has equal influence. If methods are within 10% of the best, choose the simpler explainable method.

No random row split is used because rows from the same flock history are related and would leak information across train and test sets.

In [4]:
result = train_outcome_model(dataset, "recovery")
manifest = result.manifest
print("Champion:", manifest["selected_model"])
print("Model version:", manifest["model_version"])
print("Selected X inputs:")
for feature in manifest["feature_columns"]:
    print(" -", feature)

Champion: ridge_core
Model version: recovery-0.7.0
Selected X inputs:
 - cycle_day
 - percentage_alive
 - mortality_daily_per_1000
 - mortality_recent_3d_per_1000
 - mortality_trend_delta_per_1000
 - feed_daily_per_1000_birds
 - feed_cumulative_per_1000_birds
 - temperature_recent_avg_c
 - humidity_recent_avg_pct


## 3. Candidate comparison

In [5]:
comparison = pd.DataFrame([
    {
        "Candidate": name,
        "MAE (points)": metrics["mae"] * 100,
        "Cycle-macro MAE (points)": metrics["cycle_macro_mae"] * 100,
        "RMSE (points)": metrics["rmse"] * 100,
        "Bias (points)": metrics["bias"] * 100,
        "Target-side accuracy": metrics["target_side_accuracy"],
    }
    for name, metrics in manifest["metrics"].items()
]).sort_values("Cycle-macro MAE (points)")
comparison.round({"MAE (points)": 2, "Cycle-macro MAE (points)": 2, "RMSE (points)": 2, "Bias (points)": 2, "Target-side accuracy": 3})

,Candidate,MAE (points),Cycle-macro MAE (points),RMSE (points),Bias (points),Target-side accuracy
3,random_forest,1.45,1.33,1.83,0.05,0.795
4,ridge_no_weight,1.34,1.38,1.69,0.06,0.844
5,ridge_core,1.32,1.42,1.76,-0.07,0.844
2,ridge,1.50,1.56,1.92,0.15,0.811
1,historical_mean,1.67,1.73,2.16,0.04,0.844
0,trend_naive,3.55,3.34,4.32,3.55,0.385


In [6]:
cycle_performance = pd.DataFrame.from_dict(manifest["selected_metrics"]["cycle"], orient="index")
cycle_performance.index.name = "Held-out cycle"
cycle_performance.assign(
    mae_points=cycle_performance.mae * 100,
    rmse_points=cycle_performance.rmse * 100,
    bias_points=cycle_performance.bias * 100,
)[["rows", "mae_points", "rmse_points", "bias_points"]].round(2)

,rows,mae_points,rmse_points,bias_points
Held-out cycle,,,,
2025-2,12,2.47,2.51,-2.47
2025-3,25,0.66,0.93,0.34
2025-4,25,0.91,1.10,0.06
2025-5,30,1.63,1.87,-0.52
2026-1,30,1.44,2.20,0.89


In [7]:
selected = manifest["selected_metrics"]
print(f"Selected held-out MAE: {selected['mae']*100:.2f} percentage points")
print(f"Selected held-out RMSE: {selected['rmse']*100:.2f} percentage points")
print(f"80% empirical error half-width: ±{selected['uncertainty_half_width_80']*100:.2f} points")
print(f"Target-side accuracy: {selected['target_side_accuracy']:.1%}")
print(f"Majority baseline accuracy: {selected['majority_side_accuracy']:.1%}")

Selected held-out MAE: 1.32 percentage points
Selected held-out RMSE: 1.76 percentage points
80% empirical error half-width: ±2.25 points
Target-side accuracy: 84.4%
Majority baseline accuracy: 84.4%


**Interpretation:** the compact Ridge is useful as a continuous estimate, but its target-side accuracy does not beat the majority baseline. It should be presented as a prototype projection with uncertainty—not as a proven classifier of 95% target attainment.

## 4. What the selected model relies on

In [8]:
importance = pd.DataFrame(manifest["global_feature_importance"])
importance.head(10).rename(columns={
    "feature": "Input",
    "coefficient_per_standard_deviation": "Recovery change for +1 SD",
    "absolute_importance_pct": "Share of absolute reliance (%)",
    "direction": "Direction",
}).round(4)

,Input,Recovery change for +1 SD,Share of absolute reliance (%),Direction
0,percentage_alive,0.0142,26.9175,Raises estimate
1,missing__temperature_recent_avg_c,0.0086,16.3202,Raises estimate
2,mortality_recent_3d_per_1000,-0.0052,9.7742,Lowers estimate
3,feed_cumulative_per_1000_birds,0.0050,9.4268,Raises estimate
4,cycle_day,0.0049,9.3624,Raises estimate
5,mortality_trend_delta_per_1000,0.0048,9.0937,Raises estimate
6,missing__humidity_recent_avg_pct,-0.0043,8.0718,Lowers estimate
7,mortality_daily_per_1000,-0.0032,6.1165,Lowers estimate
8,temperature_recent_avg_c,-0.0018,3.4320,Lowers estimate
9,humidity_recent_avg_pct,-0.0006,1.1747,Lowers estimate


These are standardized Ridge coefficients. They show model reliance after accounting for other inputs; they are **associations, not causal effects**. Missing-value indicators can rank highly because environmental coverage is sparse.

## 5. Day 14 held-out proof and one complete example

In [9]:
def cycle_bootstrap_mae(frame, error_column, repeats=5000, seed=42):
    # Bootstrap whole cycles, never individual rows, to preserve grouped evidence.
    rng = np.random.default_rng(seed)
    grouped = {cycle: group for cycle, group in frame.groupby("cycle_id")}
    cycles = np.array(list(grouped))
    estimates = []
    for _ in range(repeats):
        selected = rng.choice(cycles, size=len(cycles), replace=True)
        errors = np.concatenate([grouped[cycle][error_column].to_numpy(float) for cycle in selected])
        estimates.append(np.mean(np.abs(errors)))
    return np.quantile(estimates, [0.025, 0.975])

In [10]:
day14 = pd.DataFrame(manifest["day14_backtest"])
day14["error_points"] = day14["error"] * 100
day14["absolute_error_points"] = day14["absolute_error"] * 100
ci = cycle_bootstrap_mae(day14, "error_points")
metrics = manifest["day14_backtest_metrics"]
print(f"Day 14 building outcomes: {metrics['building_cycles']}")
print(f"Day 14 MAE: {metrics['mae']*100:.2f} points")
print(f"Cycle-bootstrap 95% interval for Day 14 MAE: {ci[0]:.2f} to {ci[1]:.2f} points")
example = day14.iloc[0]
print("\nExample")
print(f"Cycle/building: {example.cycle_id} / {example.building_id}")
print(f"Day 14 held-out projection: {example.predicted:.1%}")
print(f"Last-recorded actual proxy: {example.actual:.1%}")
print(f"Error = projected - actual: {example.error_points:+.2f} percentage points")
day14.head(8)[["cycle_id", "building_id", "predicted", "actual", "error_points"]]

Day 14 building outcomes: 25
Day 14 MAE: 1.43 points
Cycle-bootstrap 95% interval for Day 14 MAE: 0.92 to 1.93 points

Example
Cycle/building: 2025-2 / Tags 1
Day 14 held-out projection: 92.8%
Last-recorded actual proxy: 94.3%
Error = projected - actual: -1.46 percentage points


,cycle_id,building_id,predicted,actual,error_points
0,2025-2,Tags 1,0.928202,0.942794,-1.459237
1,2025-2,Tags 2,0.926727,0.949706,-2.297907
2,2025-2,Tags 3,0.928941,0.955588,-2.664758
3,2025-3,Lags 1,0.918556,0.900036,1.852028
4,2025-3,Lags 2,0.912343,0.907909,0.443425
5,2025-3,Tags 1,0.938303,0.938072,0.023151
6,2025-3,Tags 2,0.933551,0.931068,0.248288
7,2025-3,Tags 3,0.937977,0.944998,-0.702070


## 6. Why SMOTE or oversampling is not used

- The outcome is continuous regression, while standard SMOTE is designed for classification.
- The scarce item is the number of independent building-cycle outcomes—not the number of spreadsheet rows. Synthetic rows do not create new farms or cycles.
- Interpolating flock records could create biologically implausible combinations and falsely narrow validation error.
- Oversampling before grouped validation could leak the held-out cycle.

**Safer small-data strategy used here:** simple regularized candidates, complete-cycle holdouts, balanced checkpoints, empirical uncertainty, cycle-level bootstrap intervals, and transparent limitations. The strongest improvement is collecting more standardized completed cycles with verified harvest events.

## 7. Defense takeaway

Canary's recovery output is a **cycle-held-out Ridge estimate of the agreed last-recorded recovery proxy**. Its held-out MAE is roughly 1–2 percentage points, but it is not yet strong at recognizing the small number of cycles that finish at or above 95%. Use it to rank likely outcome gaps and guide attention, not to claim certainty.